In [6]:
import pandas as pd
import numpy as np

print("Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...")
n_rows = 50000
np.random.seed(42) # Para reproducibilidad en clase

# Distribución realista de tipos de transacción
tipos = ["PAYMENT", "TRANSFER", "CASH_OUT", "CASH_IN", "DEBIT"]
probabilidades = [0.35, 0.15, 0.30, 0.18, 0.02]

# Construimos un DataFrame de Pandas con datos aleatorios pero coherentes
df_pd = pd.DataFrame({
    "step": np.random.randint(1, 100, n_rows),
    "type": np.random.choice(tipos, n_rows, p=probabilidades),
    "amount": np.round(np.random.exponential(scale=100000, size=n_rows), 2),
    "nameOrig": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
    "oldbalanceOrg": np.round(np.random.exponential(scale=150000, size=n_rows), 2),
    "nameDest": ["C" + str(i) for i in np.random.randint(10000, 99999, n_rows)],
    "oldbalanceDest": np.round(np.random.exponential(scale=200000, size=n_rows), 2),
})

# Lógica básica de alteración de saldos
df_pd["newbalanceOrig"] = np.maximum(df_pd["oldbalanceOrg"] - df_pd["amount"], 0)
df_pd["newbalanceDest"] = df_pd["oldbalanceDest"] + df_pd["amount"]

# Inyectar Fraude (Imbalanceado): ~1% de fraude, concentrado en transferencias grandes
df_pd["isFraud"] = 0
mask_fraude = (df_pd["type"].isin(["TRANSFER", "CASH_OUT"])) & (df_pd["amount"] > 150000) & (np.random.rand(n_rows) < 0.05)
df_pd.loc[mask_fraude, "isFraud"] = 1

# Guardamos el dataset en el disco local de Colab
df_pd.to_csv("paysim_sample.csv", index=False)
print("¡Archivo 'paysim_sample.csv' creado con éxito!")

Generando archivo CSV sintético de 50,000 transacciones (esto tomará unos segundos)...
¡Archivo 'paysim_sample.csv' creado con éxito!


In [7]:
# 3. LECTURA DEL DATASET CON PYSPARK (¡Aquí empieza la clase!)
# Enseñamos a los alumnos a usar spark.read, que es como se hace en la vida real
print("\nCargando datos con PySpark SQL...")

df = spark.read.csv(
    "paysim_sample.csv",
    header=True,       # La primera fila tiene los nombres de las columnas
    inferSchema=True   # PySpark adivina el tipo de dato (entero, string, float)
)

print("Esquema inferido por Spark:")
df.printSchema()

print("Primeras 5 filas distribuidas:")
df.show(5)


Cargando datos con PySpark SQL...
Esquema inferido por Spark:
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)

Primeras 5 filas distribuidas:
+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|step|   type|   amount|nameOrig|oldbalanceOrg|nameDest|oldbalanceDest|    newbalanceOrig|newbalanceDest|isFraud|
+----+-------+---------+--------+-------------+--------+--------------+------------------+--------------+-------+
|  52|PAYMENT| 75260.18|  C22062|    302563.46|  C47999|     159891.14|227303.28000000003|     235151.32|      0|
|  93|PAYMENT|123158.12|  C264

**PySpark Machine Learning**

In [8]:
# 1. PREPARACIÓN DEL ENTORNO
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Asumimos que el 'df' anterior está cargado

In [9]:
# 2. PREPROCESAMIENTO (Pipeline de MLlib)
# Los modelos de Spark requieren que todas las variables estén en una sola columna vectorial

# Initialize SparkSession if not already active (should ideally be done once at the start of the notebook)
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PySparkML").getOrCreate()

# A. Convertir variable categórica 'type' a numérica
indexer = StringIndexer(inputCol="type", outputCol="typeIndex")

# B. One Hot Encoding para la categoría
encoder = OneHotEncoder(inputCol="typeIndex", outputCol="typeVec")

# C. Ensamblar todas las variables numéricas en un solo vector "features"
columnas_feat = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "typeVec"]
assembler = VectorAssembler(inputCols=columnas_feat, outputCol="features")

In [10]:
# 3. DIVISIÓN DEL DATASET
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# 4. DEFINICIÓN DEL MODELO
rf = RandomForestClassifier(labelCol="isFraud", featuresCol="features", numTrees=20)

# 5. CONSTRUCCIÓN DEL PIPELINE
# Esto automatiza el flujo: Indexer -> Encoder -> Assembler -> Modelo
pipeline = Pipeline(stages=[indexer, encoder, assembler, rf])

In [11]:
# 6. ENTRENAMIENTO
print("Entrenando modelo de detección de fraude...")
model = pipeline.fit(train_data)

Entrenando modelo de detección de fraude...


In [12]:
# 7. EVALUACIÓN
predictions = model.transform(test_data)

# Ver resultados de la predicción
predictions.select("amount", "type", "probability", "prediction", "isFraud").show(10)

# Medir precisión (AUC-ROC)
evaluator = BinaryClassificationEvaluator(labelCol="isFraud", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"Área bajo la curva ROC: {auc:.4f}")

+--------+-------+--------------------+----------+-------+
|  amount|   type|         probability|prediction|isFraud|
+--------+-------+--------------------+----------+-------+
| 3079.45|CASH_IN|[0.99533366390533...|       0.0|      0|
| 6302.36|CASH_IN|[0.99519606500612...|       0.0|      0|
| 6669.15|CASH_IN|[0.99384933890862...|       0.0|      0|
| 14760.3|CASH_IN|[0.99519606500612...|       0.0|      0|
|16473.44|CASH_IN|[0.99533366390533...|       0.0|      0|
|26185.66|CASH_IN|[0.99533366390533...|       0.0|      0|
|33536.78|CASH_IN|[0.99533366390533...|       0.0|      0|
|46477.14|CASH_IN|[0.99519606500612...|       0.0|      0|
|63196.67|CASH_IN|[0.99384933890862...|       0.0|      0|
|64313.39|CASH_IN|[0.99533366390533...|       0.0|      0|
+--------+-------+--------------------+----------+-------+
only showing top 10 rows
Área bajo la curva ROC: 0.8481


In [13]:
# 8. GUARDAR EL MODELO (Opcional)
# model.save("modelo_fraude_spark")

**PartitionBy**

Window.partitionBy()
from pyspark.sql.window import Window

In [14]:
# 1. Importar la clase Window
from pyspark.sql.window import Window

In [15]:
# 2. Crear dataset sintético "de juguete" (Serie Temporal)
datos = [
    ("2026-04-14", "Producto A", 100),
    ("2026-04-15", "Producto A", 120), # Sube 20
    ("2026-04-16", "Producto A", 110), # Baja 10
    ("2026-04-14", "Producto B", 50),
    ("2026-04-15", "Producto B", 55),  # Sube 5
    ("2026-04-16", "Producto B", 60)   # Sube 5
]

columnas = ["Fecha", "Producto", "Ventas"]
df = spark.createDataFrame(datos, columnas)
df.show()

+----------+----------+------+
|     Fecha|  Producto|Ventas|
+----------+----------+------+
|2026-04-14|Producto A|   100|
|2026-04-15|Producto A|   120|
|2026-04-16|Producto A|   110|
|2026-04-14|Producto B|    50|
|2026-04-15|Producto B|    55|
|2026-04-16|Producto B|    60|
+----------+----------+------+



In [16]:
# 3. Definir una Ventana o Partición
# Se agrupan registros según producto, sin llegar a realizar agregaciones
# Se ordenan por fecha dentro de cada producto
ventana_producto = Window.partitionBy("Producto").orderBy("Fecha")

**IMPORTANTE**: Podría parecer que el antes del paso 3, los registros del Dataset ya estaban organizados tal y como se especifica en `Window.partitionBy().orderBy()`, y que por tanto esta operación no supone ningún cambio.

Sin embargo, lo que realmente hace esta operación es decirle a Spark cómo organizar los datos en particiones (Shuffle). Spark por defecto realizaría particiones internamente para el procesamiento distribuido de un gran dataset.

En una serie temporal, si dejamos a Spark el libre albedrío de particionar "a su manera", se perdería el sentido del orden temporal, y esto no nos permitiría usar operaciones básicas de series temporales como `lag()`, que obtiene valores del registro (ej. día) anterior: se perdería esta noción al estar los datos desordenados en los distintos nodos del cluster.

Así, tenemos que:

*   `partitionBy()`: busca en todos los nodos (procesadores) los registros de un tipo (ej. producto) determinado, y los coloca todos en un mismo procesador.
* `orderBy()`: una vez los registros de un mismo tipo están juntos en un mismo nodo, se ordenan para recuperar el sentido del orden.

In [19]:
# 4. Aplicar la función lag() de obtener valores de retardo usando nuestra ventana
# lag(columna, 1) obtendrá el valor de la fila inmediatamente anterior
from pyspark.sql import functions as F
df_resultado = df.withColumn(
    "Ventas_Ayer",
    F.lag("Ventas", 1).over(ventana_producto)
).withColumn(
    "Crecimiento_Diario",
    F.col("Ventas") - F.col("Ventas_Ayer")
)

# df.withColumn(nombre_columna, valor_columna) --> crea una nueva columna en df

# Mostrar el resultado
df_resultado.show()

+----------+----------+------+-----------+------------------+
|     Fecha|  Producto|Ventas|Ventas_Ayer|Crecimiento_Diario|
+----------+----------+------+-----------+------------------+
|2026-04-14|Producto A|   100|       NULL|              NULL|
|2026-04-15|Producto A|   120|        100|                20|
|2026-04-16|Producto A|   110|        120|               -10|
|2026-04-14|Producto B|    50|       NULL|              NULL|
|2026-04-15|Producto B|    55|         50|                 5|
|2026-04-16|Producto B|    60|         55|                 5|
+----------+----------+------+-----------+------------------+



In [20]:
print("\nDataFrame con valores NULL rellenados y Crecimiento_Diario recalculado:")
df_temp = df_resultado.fillna(0, subset=["Ventas_Ayer"]) # Rellenar solo Ventas_Ayer
df_final = df_temp.withColumn(
    "Crecimiento_Diario",
    F.col("Ventas") - F.col("Ventas_Ayer") # Recalcular Crecimiento_Diario
)
df_final.show()


DataFrame con valores NULL rellenados y Crecimiento_Diario recalculado:
+----------+----------+------+-----------+------------------+
|     Fecha|  Producto|Ventas|Ventas_Ayer|Crecimiento_Diario|
+----------+----------+------+-----------+------------------+
|2026-04-14|Producto A|   100|          0|               100|
|2026-04-15|Producto A|   120|        100|                20|
|2026-04-16|Producto A|   110|        120|               -10|
|2026-04-14|Producto B|    50|          0|                50|
|2026-04-15|Producto B|    55|         50|                 5|
|2026-04-16|Producto B|    60|         55|                 5|
+----------+----------+------+-----------+------------------+

